# ☀️ Photovoltaic Panel 1: Expected Power & Digital Twin
### Model Target: Expected Power ($P_1$) using Environmental Inputs ($Irr$ and $T_1$) Only

---

### 📋 Overview & Methodology:
1. **Inputs**: Strictly Solar Irradiance (`Irr`) [W/m²] and Panel Temperature (`T1`) [°C].
   - *Note*: We explicitly do not use electrical signals ($V_1, I_1$) as model inputs, because under faults (e.g., shading, degradation, open circuit) electrical outputs collapse. The Expected Power represents what a **healthy panel ought to generate** under current atmospheric conditions.
2. **Feature Engineering**: Incorporates physics-informed features:
   - Interaction term: $Irr \times T_1$
   - Quadratic terms: $Irr^2$, $T_1^2$
   - Temperature derating: $Irr \times (1 - 0.0045 \times (T_1 - 25.0))$ (capturing the silicon temperature coefficient)
   - Irradiance per Kelvin: $Irr / (T_1 + 273.15)$
3. **Training on Normal Baseline**:
   - Model is trained exclusively on **active healthy conditions** (`Label_S1 == 'Normal'` and $V_1 \ge 5.0\text{V}$).
4. **Evaluating on Normal + Anomalies**:
   - The test set reserves unseen **Normal** cases alongside all **Anomalies** (`Degradation`, `Shading`, `Open Circuit`, `Short Circuit`) to benchmark fault detection via **Residual Power Loss** and **Performance Ratio ($PR\%$)**.


In [ ]:
import os
import sys
from pathlib import Path

# Ensure UTF-8 output encoding across environments
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

print("Environment and libraries loaded successfully.")


## 1. Feature Engineering (Strictly using `Irr` and `T1`)
We derive non-linear, interaction, and thermodynamic temperature derating terms strictly from solar irradiance and panel temperature.


In [ ]:
def extract_pv_features(df_input: pd.DataFrame) -> pd.DataFrame:
    """
    Extracts features for Expected Power prediction using ONLY Irradiance and Temperature.
    
    Includes:
    - Raw Irradiance (Irr) [W/m^2]
    - Raw Panel Temperature (T1) [deg C]
    - Interaction term: Irr * T1
    - Non-linear terms: Irr^2, T1^2
    - Temperature-derated effective irradiance: Irr * (1 - 0.0045 * (T1 - 25))
    - Temperature ratio in Kelvin: Irr / (T1 + 273.15)
    """
    X = pd.DataFrame(index=df_input.index)
    irr = df_input['Irr'].astype(float)
    t1 = df_input['T1'].astype(float)
    
    X['Irr'] = irr
    X['T1'] = t1
    X['Irr_T1_interaction'] = irr * t1
    X['Irr_squared'] = irr ** 2
    X['T1_squared'] = t1 ** 2
    
    # Standard silicon PV temperature coefficient derating (-0.45% / deg C from 25 deg C)
    X['Irr_temp_derated'] = irr * (1.0 - 0.0045 * (t1 - 25.0))
    X['Irr_per_Kelvin'] = irr / (t1 + 273.15)
    
    return X


## 2. Load Dataset and Daylight Telemetry
Loads `PV_Diagnostic_Cleaned.csv`, filters for active daytime records ($Irr \ge 50\text{ W/m}^2$), and prepares train/test splits.


In [ ]:
def load_and_prepare_dataset(csv_path: str):
    """
    Loads dataset, isolates active daytime telemetry, and creates train/test splits.
    """
    print(f"Loading dataset from: {csv_path}")
    df = pd.read_csv(csv_path, sep=';')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Filter daylight hours (Irr >= 50 W/m^2) where PV power generation is active
    day_df = df[df['Irr'] >= 50.0].copy().reset_index(drop=True)
    print(f"Total records: {len(df):,} | Daytime records (Irr >= 50 W/m²): {len(day_df):,}")
    
    # Normal active generation condition (Panel healthy and producing power)
    normal_mask = (day_df['Label_S1'] == 'Normal') & (day_df['V1'] >= 5.0)
    normal_df = day_df[normal_mask].copy()
    anomaly_df = day_df[day_df['Label_S1'] != 'Normal'].copy()
    
    # 80/20 train/test split on NORMAL data
    norm_train_df, norm_test_df = train_test_split(
        normal_df, test_size=0.20, random_state=42
    )
    
    # Combined Test Set containing BOTH unseen Normal states and Anomalies
    test_combined_df = pd.concat([norm_test_df, anomaly_df], axis=0).reset_index(drop=True)
    
    print(f"\n--- Split Summary ---")
    print(f"Training set (Normal Active only)     : {len(norm_train_df):,} samples")
    print(f"Test set (Unseen Normal + Anomalies) : {len(test_combined_df):,} samples")
    print(f"  - Unseen Normal samples in test     : {len(norm_test_df):,} samples")
    print(f"  - Anomalous samples in test         : {len(anomaly_df):,} samples")
    
    return norm_train_df, norm_test_df, test_combined_df, day_df

# Candidates path detection
candidates = [
    Path("data/PV_Diagnostic_Cleaned.csv"),
    Path("../data/PV_Diagnostic_Cleaned.csv"),
    Path(r"C:\Users\kirok\Desktop\New folder (7)\data\PV_Diagnostic_Cleaned.csv"),
]
dataset_file = next((str(p) for p in candidates if p.exists()), None)
if dataset_file is None:
    raise FileNotFoundError("Could not find PV_Diagnostic_Cleaned.csv")

train_norm, test_norm, test_combined, full_day = load_and_prepare_dataset(dataset_file)


## 3. Train XGBoost Model on Normal Healthy Telemetry
We train exclusively on the healthy baseline to learn the mapping $(Irr, T_1) \to P_{1,\text{expected}}$, then evaluate on unseen normal test data.


In [ ]:
def train_expected_power_model(train_df: pd.DataFrame, val_df: pd.DataFrame):
    """
    Trains an XGBoost Regressor on the normal condition to learn P1 = f(Irr, T1).
    """
    X_train = extract_pv_features(train_df)
    y_train = train_df['P1'].astype(float)
    
    X_val = extract_pv_features(val_df)
    y_val = val_df['P1'].astype(float)
    
    print("Training XGBoost Expected Power Model (Normal Condition Baseline)...")
    model = XGBRegressor(
        n_estimators=250,
        max_depth=7,
        learning_rate=0.06,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    
    # Validation evaluation on normal healthy data
    y_pred_val = np.maximum(0.0, model.predict(X_val))
    r2 = r2_score(y_val, y_pred_val)
    mae = mean_absolute_error(y_val, y_pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    
    print("\n=======================================================")
    print("      Model Performance on Unseen Normal Test Data      ")
    print("=======================================================")
    print(f"  R² Score (Coefficient of Determination) : {r2:.4f}")
    print(f"  Mean Absolute Error (MAE)               : {mae:.2f} Watts")
    print(f"  Root Mean Squared Error (RMSE)          : {rmse:.2f} Watts")
    print("=======================================================\n")
    
    return model

model = train_expected_power_model(train_norm, test_norm)

# Save trained artifact
models_dir = Path("../models") if Path("../models").exists() else Path("models")
models_dir.mkdir(parents=True, exist_ok=True)
saved_model_path = models_dir / "pv_expected_power_p1_xgb.pkl"
joblib.dump(model, saved_model_path)
print(f"Model artifact saved to: {saved_model_path}")


## 4. Interactive Prediction: Pass `(Irr, T1)` -> Predict Expected Power
Function to predict expected healthy power from arbitrary solar irradiance ($W/m^2$) and temperature ($^\circ\text{C}$).


In [ ]:
def predict_expected_power(model, irradiance: float, temperature: float) -> float:
    """
    Given single values of Irradiance (W/m^2) and Temperature (deg C),
    predicts the expected healthy power (Watts) for Panel 1.
    """
    sample_df = pd.DataFrame({'Irr': [irradiance], 'T1': [temperature]})
    X_sample = extract_pv_features(sample_df)
    expected_p = float(model.predict(X_sample)[0])
    return max(0.0, expected_p)

# Interactive demonstration
demo_scenarios = [
    {"Irr": 200.0, "T1": 25.0, "desc": "Low morning sunlight, cool panel"},
    {"Irr": 600.0, "T1": 38.0, "desc": "Moderate midday sunlight, warm panel"},
    {"Irr": 850.0, "T1": 42.0, "desc": "Strong afternoon sunlight, hot panel"},
    {"Irr": 1000.0, "T1": 45.0, "desc": "Peak STC irradiance, high temperature"}
]

print("--- Example Predictions: See (Irr, T1) -> Expected Power ---")
for sc in demo_scenarios:
    p_exp = predict_expected_power(model, sc['Irr'], sc['T1'])
    print(f"  • {sc['desc']}:")
    print(f"    Irr = {sc['Irr']:>6.1f} W/m², T1 = {sc['T1']:>4.1f}°C  -->  Expected Power P1 = {p_exp:>6.2f} Watts\n")


## 5. Full Evaluation Across Normal States and Anomalies
Evaluate the full combined test set to compute:
- **Expected Power ($P_{1,\text{expected}}$)**
- **Residual Loss ($\Delta P = P_{1,\text{expected}} - P_{1,\text{actual}}$)**
- **Performance Ratio ($PR\% = \frac{P_{1,\text{actual}}}{P_{1,\text{expected}}} \times 100\%$)**


In [ ]:
def evaluate_across_states(model, test_combined_df: pd.DataFrame):
    """
    Evaluates expected power vs actual power across Normal states and all Anomalies.
    Calculates Residual Loss and Performance Ratio (PR%).
    """
    X_test = extract_pv_features(test_combined_df)
    test_combined_df['P1_expected'] = np.maximum(0.0, model.predict(X_test))
    
    # Residual Power Loss: What the panel SHOULD produce minus what it ACTUALLY produces
    test_combined_df['Residual_Loss'] = test_combined_df['P1_expected'] - test_combined_df['P1']
    
    # Performance Ratio (PR%): (Actual / Expected) * 100%
    test_combined_df['PR%'] = np.clip(
        (test_combined_df['P1'] / np.maximum(0.1, test_combined_df['P1_expected'])) * 100.0,
        0.0, 150.0
    )
    
    # Aggregated breakdown table
    breakdown = test_combined_df.groupby('Label_S1').agg(
        Sample_Count=('P1', 'count'),
        Mean_Irr=('Irr', 'mean'),
        Mean_T1=('T1', 'mean'),
        Mean_Actual_P1=('P1', 'mean'),
        Mean_Expected_P1=('P1_expected', 'mean'),
        Mean_Residual_Loss=('Residual_Loss', 'mean'),
        Mean_PR_Percent=('PR%', 'mean')
    ).round(2)
    
    print("=========================================================================================")
    print("                    Breakdown Across Normal and Anomalous Operating States               ")
    print("=========================================================================================")
    print(breakdown)
    print("=========================================================================================\n")
    
    return test_combined_df, breakdown

evaluated_df, breakdown_stats = evaluate_across_states(model, test_combined)
breakdown_stats


## 6. Comprehensive Diagnostic Visualizations


In [ ]:
def generate_visualizations(model, test_combined_df: pd.DataFrame, save_dir: str = "images"):
    """
    Generates plots illustrating Irradiance, Temperature, and Expected vs Actual Power.
    """
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor('#f8fafc')
    
    # Plot 1: Expected Power vs Irradiance, colored by Temperature
    ax1 = axes[0, 0]
    sample_normal = test_combined_df[test_combined_df['Label_S1'] == 'Normal'].sample(n=min(3000, len(test_combined_df)), random_state=42)
    scatter1 = ax1.scatter(
        sample_normal['Irr'], sample_normal['P1_expected'],
        c=sample_normal['T1'], cmap='coolwarm', alpha=0.6, s=15, edgecolors='none'
    )
    cbar1 = plt.colorbar(scatter1, ax=ax1)
    cbar1.set_label('Temperature T1 (°C)', fontsize=10)
    ax1.set_title('Digital Twin: Expected Power vs Irradiance & Temperature', fontsize=12, fontweight='bold', pad=10)
    ax1.set_xlabel('Solar Irradiance Irr (W/m²)', fontsize=11)
    ax1.set_ylabel('Predicted Expected Power P1 (Watts)', fontsize=11)
    ax1.set_facecolor('#ffffff')

    # Plot 2: Actual vs Expected Power Scatter (Normal vs Anomalies)
    ax2 = axes[0, 1]
    palette = {
        'Normal': '#10b981',       # Green
        'Degradation': '#f59e0b',  # Amber
        'Shading': '#3b82f6',      # Blue
        'Open Circuit': '#ef4444', # Red
        'Short Circuit': '#8b5cf6' # Purple
    }
    
    # Subsample for clear visualization
    sample_vis = test_combined_df.groupby('Label_S1', group_keys=False).apply(
        lambda g: g.sample(min(len(g), 800), random_state=42),
        include_groups=True
    )
    
    for label, group in sample_vis.groupby('Label_S1'):
        ax2.scatter(
            group['P1_expected'], group['P1'],
            label=label, alpha=0.5, s=16, color=palette.get(label, '#6b7280')
        )
    
    # 1:1 Ideal Reference Line
    max_val = max(sample_vis['P1_expected'].max(), sample_vis['P1'].max())
    ax2.plot([0, max_val], [0, max_val], 'k--', lw=1.5, label='Ideal Healthy (Actual = Expected)')
    ax2.set_title('Actual Power vs Expected Power by Operating Mode', fontsize=12, fontweight='bold', pad=10)
    ax2.set_xlabel('Expected Power P1 (Watts)', fontsize=11)
    ax2.set_ylabel('Actual Measured Power P1 (Watts)', fontsize=11)
    ax2.legend(frameon=True, facecolor='#ffffff')
    ax2.set_facecolor('#ffffff')

    # Plot 3: Performance Ratio (PR%) Boxplot across Operating Modes
    ax3 = axes[1, 0]
    sns.boxplot(
        data=sample_vis, x='Label_S1', y='PR%', hue='Label_S1',
        palette=palette, ax=ax3, showfliers=False, legend=False
    )
    ax3.axhline(100, color='gray', linestyle=':', lw=1.2, label='100% PR Reference')
    ax3.set_title('Performance Ratio (PR = Actual / Expected %) across States', fontsize=12, fontweight='bold', pad=10)
    ax3.set_xlabel('Operating State', fontsize=11)
    ax3.set_ylabel('Performance Ratio (%)', fontsize=11)
    ax3.tick_params(axis='x', rotation=15)
    ax3.set_facecolor('#ffffff')

    # Plot 4: Mean Residual Power Loss by State
    ax4 = axes[1, 1]
    res_summary = test_combined_df.groupby('Label_S1')['Residual_Loss'].mean().reset_index()
    bars = ax4.bar(
        res_summary['Label_S1'], res_summary['Residual_Loss'],
        color=[palette.get(l, '#6b7280') for l in res_summary['Label_S1']],
        edgecolor='#333333', linewidth=0.8, width=0.55
    )
    for bar in bars:
        h = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width() / 2, h + 1.0, f"{h:.1f} W", ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax4.set_title('Mean Residual Power Loss (Expected - Actual) in Watts', fontsize=12, fontweight='bold', pad=10)
    ax4.set_xlabel('Operating State', fontsize=11)
    ax4.set_ylabel('Power Deficit (Watts)', fontsize=11)
    ax4.tick_params(axis='x', rotation=15)
    ax4.set_facecolor('#ffffff')
    
    plt.tight_layout()
    plot_file = os.path.join(save_dir, "panel1_expected_power_analysis.png")
    plt.savefig(plot_file, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"[SUCCESS] Visualization figure saved to: {plot_file}")
    return plot_file

plot_path = generate_visualizations(model, evaluated_df)
